# Multi-agent supervisor

A LangGraph supervisor delegating to two specialists to answer:

> What are the key government revenue streams, and how will the Budget for
> the Future Energy Fund be supported?

## The graph

```mermaid
flowchart TD
    START([START]) --> SUP{supervisor}

    SUP -->|revenue_agent| REV[revenue agent<br/>pages 9, 13, 15]
    SUP -->|expenditure_agent| EXP[expenditure agent<br/>pages 16, 18, 20]
    SUP -->|synthesis| SYN[synthesis<br/>combines findings]
    SUP -->|out_of_scope| DEC[decline<br/>fixed message]

    REV -.->|finding| SUP
    EXP -.->|finding| SUP

    SYN --> END([END])
    DEC --> END
```

Solid arrows are the supervisor's routing choices; dashed arrows are agents
returning their findings. The loop is what makes each turn a decision: after an
agent reports, the supervisor chooses again with that finding in hand.

Agents route unconditionally back to the supervisor, which is the only node
that decides. A fixed chain would have no decision to trace, and the trace is
what is wanted.

## Who reads what

| Agent | Pages | Why those |
|---|---|---|
| revenue_agent | 9, 13, 15 | Revenue breakdown chart, FY2024 narrative, NIRC |
| expenditure_agent | 16, 18, 20 | Table 2.1, top-ups prose, Table 2.4 |

**Page 13 is scoped to revenue only, deliberately.** It carries both the
revenue total and the top-ups sentence. Keeping it out of the expenditure set
means neither agent can answer the combined query alone, so the collaboration
is structural rather than staged.

Scoping is also the cost control: the whole document is ~14.8k tokens against
~1.6k for an agent's page set.

## The answer that matters

The document never states which revenue stream funds the Future Energy Fund.
Government revenue is not earmarked to particular funds. An answer saying
"funded by GST" is wrong however fluent - and it is the most likely failure
mode here, since synthesis is the one node writing free prose.

Assumptions are at the end.

## Summary: the required query

**Question:** What are the key government revenue streams, and how will the
Budget for the Future Energy Fund be supported?

| # | Figure | Value | Page |
|---|---|---|---|
| 1 | Corporate Income Tax | 27.2% of Operating Revenue | 9 |
| 2 | Personal Income Tax | 16.8% of Operating Revenue | 9 |
| 3 | Goods and Services Tax | 15.7% of Operating Revenue | 9 |
| 4 | Estimated FY2024 Operating Revenue | $108.6 billion | 13 |
| 5 | Estimated FY2024 NIRC | $23.5 billion | 15 |
| 6 | Initial injection for the Future Energy Fund | $5,000.0 million | 18 |

**Answer:** The key government revenue streams are Corporate Income Tax,
Personal Income Tax, Goods and Services Tax, Other Taxes, Stamp Duty, Assets
Taxes, Vehicle Quota Premiums, Fees and Charges, Customs, Excise and Carbon
Taxes, Betting Taxes, Motor Vehicle Taxes, Withholding Tax, Statutory Boards'
Contributions, and Others.

The document does not identify any specific revenue stream as funding the
Future Energy Fund. The Initial injection for the Future Energy Fund is $5.0
billion (p.18), which is part of the overall Budget, funded by the revenue
streams above rather than any one of them in particular.

Routing: `revenue_agent` -> `expenditure_agent` -> `synthesis` (3 turns, 1
override). Scored 5/5 on all five checks.

**Full output** (`trace.render()`, cell below):

```
QUERY: What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     revenue_agent      revenue_agent      The query has two parts: government revenue stre
2     expenditure_agent  expenditure_agent  The question about key government revenue stream
3     out_of_scope       synthesis          The key government revenue streams are unanswere
                         OVERRIDDEN:        findings were gathered, so the document does answer this

AGENT FINDINGS
agent                pages            figures
revenue_agent        9, 13, 15        5
expenditure_agent    16, 18, 20       1

CITATIONS
       27.20 percent  p.9    Corporate Income Tax
                             "Corporate Income Tax 27.2%"
       16.80 percent  p.9    Personal Income Tax
                             "Personal Income Tax 16.8%"
       15.70 percent  p.9    Goods and Services Tax
                             "Goods and Services Tax 15.7%"
      108.60 billion  p.13   Estimated FY2024 Operating Revenue
                             "Estimated FY2024 Operating Revenue is $108.6 billion"
       23.50 billion  p.15   Estimated FY2024 NIRC
                             "Estimated FY2024 NIRC is $23.5 billion"
    5,000.00 million  p.18   Initial injection for the Future Energy Fund
                             "the Government will establish the Future Energy Fund with an ini"

ANSWER
The key government revenue streams are Corporate Income Tax, Personal Income Tax, Goods and Services Tax, Other Taxes, Stamp Duty, Assets Taxes, Vehicle Quota Premiums, Fees and Charges, Customs, Excise and Carbon Taxes, Betting Taxes, Motor Vehicle Taxes, Withholding Tax, Statutory Boards' Contributions, and Others.

The Budget for the Future Energy Fund will be supported by the overall government revenue. The key components of the government revenue are:

- Corporate Income Tax: 27.2 percent (p.9)
- Personal Income Tax: 16.8 percent (p.9)
- Goods and Services Tax: 15.7 percent (p.9)
- Estimated FY2024 Operating Revenue: $108.6 billion (p.13)
- Estimated FY2024 NIRC: $23.5 billion (p.15)

The document does not identify any specific revenue stream as funding the Future Energy Fund. The Initial injection for the Future Energy Fund is $5.0 billion (p.18), which is part of the overall Budget, funded by the revenue streams mentioned above.

NODE COSTS
node                   seconds
supervisor                0.58
revenue_agent             0.78
supervisor                0.45
expenditure_agent         6.12
supervisor               10.65
synthesis                 9.71
------------------------------
total                    28.29

3 decisions, 2 agents invoked, 1 override, 6 figures cited
```

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.config import load_config
from src.graph.evaluation import load_demo_queries, score_part3
from src.graph.workflow import build_graph, run_query, stream_trace
from src.ingestion.download import ensure_pdf
from src.ingestion.parser import extract_pages
from src.llm import get_chat_model

config = load_config(Path.cwd().parent / "config.yml")
pdf_path = ensure_pdf(config.pdf_url)
model = get_chat_model(config)

print(f"provider={config.provider}  model={config.model}")
print(f"revenue pages     {config.pages_for_agent('revenue')}")
print(f"expenditure pages {config.pages_for_agent('expenditure')}")
print(f"max turns         {config.max_turns}")

provider=groq  model=llama-3.1-8b-instant
revenue pages     [9, 13, 15]
expenditure pages [16, 18, 20]
max turns         4


## The graph, as compiled

Node names and edges, read from the compiled graph rather than described.

In [2]:
graph = build_graph(model, config, pdf_path)
topology = graph.get_graph()

print("nodes:")
for node in topology.nodes:
    print(f"  {node}")

print("\nedges:")
for edge in topology.edges:
    label = f"  [{edge.data}]" if edge.data else ""
    print(f"  {edge.source:20s} -> {edge.target}{label}")

nodes:
  __start__
  supervisor
  revenue_agent
  expenditure_agent
  synthesis
  decline
  __end__

edges:
  __start__            -> supervisor
  expenditure_agent    -> supervisor
  revenue_agent        -> supervisor
  supervisor           -> decline  [out_of_scope]
  supervisor           -> expenditure_agent
  supervisor           -> revenue_agent
  supervisor           -> synthesis
  decline              -> __end__
  synthesis            -> __end__


## How the supervisor decides

It returns a structured decision. `reasoning` is declared before `next` in the
schema, so the model states why before it commits:

```python
class RouteDecision(BaseModel):
    reasoning: str
    next: Literal["revenue_agent", "expenditure_agent", "synthesis", "out_of_scope"]
    sub_task: str
```

Four deterministic guards then wrap that choice, in code rather than in the
prompt:

| Guard | Why |
|---|---|
| No agent runs twice | Its pages have not changed; a second pass reads identical text |
| Turn cap of 4 | The graph provably terminates |
| No synthesis before any finding | Synthesising nothing produces an ungrounded answer |
| No declining once findings exist | `out_of_scope` asserts the document cannot answer, which the findings contradict |

When a guard fires, the trace records both what the model chose and what ran.
A forced route is never presented as a decision.

This is the same split as Part 2: the model where judgement is needed, code
where there is one right answer.

## The required query, streamed

Each node's update as it happens, so the routing is visible while it runs.

In [3]:
REQUIRED = (
    "What are the key government revenue streams, and how will the Budget "
    "for the Future Energy Fund be supported?"
)

stream_trace(REQUIRED, model=model, config=config, pdf_path=pdf_path)

[supervisor] turn 1: revenue_agent
             The query has two parts: government revenue streams and the Budget for the Future Energy Fund. The revenue_agent covers government revenue streams, and the expenditure_agent covers government spending, including funds and top-ups.


[revenue_agent] read pages [9, 13, 15], 5 figures


[supervisor] turn 2: expenditure_agent
             The question about key government revenue streams is already answered, but the Budget for the Future Energy Fund's support is still unanswered, which falls under the expenditure agent's subject.


[expenditure_agent] read pages [16, 18, 20], 1 figures


[supervisor] turn 3: out_of_scope -> synthesis
             The key government revenue streams are unanswered, and the revenue_agent covers this subject.


[synthesis] answered


## The full trace

The same run, recorded rather than streamed: every routing decision with its
reasoning, what each agent read, every figure with the text it came from, and
what the run cost.

In [4]:
import time

# The streamed run above just used ~7k tokens; the free tier allows 6k per
# minute, so wait before running the same query again.
time.sleep(60)

trace = run_query(REQUIRED, model=model, config=config, pdf_path=pdf_path)
print(trace.render())

QUERY: What are the key government revenue streams, and how will the Budget for the Future Energy Fund be supported?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     revenue_agent      revenue_agent      The query has two parts: government revenue stre
2     expenditure_agent  expenditure_agent  The question about key government revenue stream
3     out_of_scope       synthesis          The key government revenue streams are unanswere
                         OVERRIDDEN:        findings were gathered, so the document does answer this

AGENT FINDINGS
agent                pages            figures
revenue_agent        9, 13, 15        5
expenditure_agent    16, 18, 20       1

CITATIONS
       27.20 percent  p.9    Corporate Income Tax
                             "Corporate Income Tax 27.2%"
       16.80 percent  p.9    Personal Income Tax
                             "

## Scoring an open-ended answer

Prose has no single correct wording, so it is not scored. Five things are:

| Check | What it catches |
|---|---|
| Routing | An agent that should have run and did not, or one that ran unnecessarily |
| Figures | A wrong value, or the right value with the wrong unit - 5000 billions is a 1000x error |
| Traceability | A quote that does not appear on the page it cites |
| Page discipline | A figure from a page the agent was never given |
| Labels | A cited figure described differently from how its finding labelled it, e.g. renaming Operating Revenue to "total government revenue" |

The third is the strongest: the model can read the right sentence and cite the
wrong page, leaving a correct value that cannot be verified.

The required query passes all five checks - see the score below. Across the
full demo set (next section), the one check that fails is routing, on
`nirc_classification`: `expenditure_agent` runs and reports zero figures,
costing an extra turn without changing the answer. See "Where the checks
failed" for detail.

In [5]:
def page_text_for(trace):
    """The text of every page this trace cites, for verifying quotes."""
    pages = {citation.page for citation in trace.citations()}
    return {
        page: extract_pages(pdf_path, [page]) for page in sorted(pages)
    }


report = score_part3(
    trace,
    expected={
        "routed_to": ["revenue_agent", "expenditure_agent"],
        "figures": [{"value": 108.6, "unit": "billion"}],
        "figures_any_of": [
            [{"value": 5.0, "unit": "billion"}, {"value": 5000, "unit": "million"}]
        ],
    },
    config=config,
    page_text=page_text_for(trace),
)

print(report.table())

field                        result detail
------------------------------------------------------------------------------
routing                      Pass   invoked revenue_agent, expenditure_agent
figures                      Pass   2 required figure(s) present, units correct
traceability                 Pass   6/6 quotes found on their cited page
page discipline              Pass   every figure came from an agent's own pages
labels                       Pass   every cited figure keeps its finding's wording

5/5 checks passed


## Every demo query

Queries live in `expectations/demo_queries.yaml`, so one can be added, reworded
or disabled without touching code.

Together they show single-agent routing both ways, two agents collaborating,
one agent's findings informing the other, and two queries declined.

In [6]:
queries = load_demo_queries()

for query in queries:
    print(f"{query['id']:24s} {query['routed_to'] or ['decline']}")
    print(f"{'':24s} {query['demonstrates'].strip()[:90]}")
    print()

revenue_only             ['revenue_agent']
                         Single-agent routing. The supervisor sends this to revenue alone and does not invoke expen

expenditure_only         ['expenditure_agent']
                         Single-agent routing the other way, and the unit trap - the same amount appears as 5.00 bi

required                 ['revenue_agent', 'expenditure_agent']
                         The two-part query. Two agents collaborating - neither can answer it alone, because page 1

collaboration            ['revenue_agent', 'expenditure_agent']
                         One agent's findings informing the other. The expenditure pages say what the top-ups are; 

nirc_classification      ['revenue_agent']
                         A question needing the document read rather than pattern-matched. NIRC is part of Total Re

out_of_scope_sensible    ['decline']
                         Declining a question the document cannot answer. The sharpest case - the model knows the a



In [7]:
import time

# Groq's free tier allows 6,000 tokens per minute. A full query is ~7k across
# six calls, so running several back to back trips the limit. Pausing between
# queries keeps the run inside it; the wait is the constraint, not the work.
PAUSE_SECONDS = 45

results = {}

for index, query in enumerate(queries):
    if index:
        time.sleep(PAUSE_SECONDS)

    trace_for_query = run_query(
        query["query"], model=model, config=config, pdf_path=pdf_path
    )
    report_for_query = score_part3(
        trace_for_query,
        expected=query,
        config=config,
        page_text=page_text_for(trace_for_query),
    )
    results[query["id"]] = (trace_for_query, report_for_query)

    print(f"{query['id']:24s} {report_for_query.summary():22s} {trace_for_query.summary()}")

revenue_only             5/5 checks passed      2 decisions, 1 agent invoked, 1 override, 5 figures cited


expenditure_only         5/5 checks passed      2 decisions, 1 agent invoked, 1 override, 1 figure cited


required                 5/5 checks passed      3 decisions, 2 agents invoked, 1 override, 6 figures cited


collaboration            5/5 checks passed      3 decisions, 2 agents invoked, 1 override, 12 figures cited


nirc_classification      4/5 checks passed      3 decisions, 2 agents invoked, 0 overrides, 3 figures cited


out_of_scope_sensible    5/5 checks passed      1 decision, 0 agents invoked, 0 overrides, 0 figures cited


out_of_scope_nonsense    5/5 checks passed      1 decision, 0 agents invoked, 0 overrides, 0 figures cited


### Where the checks failed

The interesting output. Each failure is a real observation about the
system's behaviour, not a defect in the check.


In [8]:
# Which checks failed, and why. Reads the traces already gathered above -
# no further model calls.
for query in queries:
    trace_for_query, report_for_query = results[query["id"]]
    failed = [check for check in report_for_query.checks if not check.passed]
    if not failed:
        continue

    print(f"{query['id']}")
    for check in failed:
        print(f"    {check.field:18s} {check.detail[:96]}")
    for decision in trace_for_query.decisions:
        if decision.was_overridden:
            print(f"    override         turn {decision.turn}: {decision.chose} -> "
                  f"{decision.routed_to} ({decision.overridden})")
    print()

nirc_classification
    routing            invoked unnecessarily: expenditure_agent


### Routing across the demo set

The same graph takes a different path for each query. Nothing hardcodes the
order - the supervisor picks each turn, and after each agent it picks again
with that agent's findings in hand.

In [9]:
print(f"{'query':24s} {'expected':34s} {'actual':34s} turns")
print("-" * 100)
for query in queries:
    trace_for_query, _ = results[query["id"]]
    expected = ", ".join(query["routed_to"]) or "decline"
    actual = ", ".join(trace_for_query.agents_invoked) or "decline"
    print(
        f"{query['id']:24s} {expected:34s} {actual:34s} "
        f"{len(trace_for_query.decisions)}"
    )

query                    expected                           actual                             turns
----------------------------------------------------------------------------------------------------
revenue_only             revenue_agent                      revenue_agent                      2
expenditure_only         expenditure_agent                  expenditure_agent                  2
required                 revenue_agent, expenditure_agent   revenue_agent, expenditure_agent   3
collaboration            revenue_agent, expenditure_agent   expenditure_agent, revenue_agent   3
nirc_classification      revenue_agent                      revenue_agent, expenditure_agent   3
out_of_scope_sensible    decline                            decline                            1
out_of_scope_nonsense    decline                            decline                            1


### The collaboration query

"The Government tops up Endowment and Trust Funds by $20.4 billion. Where does
the money come from?"

Neither agent can answer this alone. The expenditure pages say what the top-ups
are; only the revenue pages say what funds the Budget they come from. Whichever
agent runs second sees the first's findings in the supervisor's next prompt.

In [10]:
if "collaboration" in results:
    print(results["collaboration"][0].render())

QUERY: The Government tops up Endowment and Trust Funds by $20.4 billion. Where does the money come from?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     expenditure_agent  expenditure_agent  The question asks where the money comes from, wh
2     revenue_agent      revenue_agent      The question asks where the money for the top-up
3     out_of_scope       synthesis          The question asks where the money for Fund top-u
                         OVERRIDDEN:        findings were gathered, so the document does answer this

AGENT FINDINGS
agent                pages            figures
expenditure_agent    16, 18, 20       11
revenue_agent        9, 13, 15        1

CITATIONS
   20,400.00 million  p.18   Total top-ups to Endowment and Trust Funds
                             "In total, $20.4 billion will be committed to Fund top-ups (see T"
    6,000.00 million  p.20   

### A declined query

"What is the capital of France?" is the sharpest test of grounding: the model
knows the answer from training, so answering would be ungrounded by definition.

The supervisor routes to `out_of_scope` and the decline is a fixed message, not
a model call - there is no opportunity for it to answer from memory. Both
`out_of_scope_sensible` and `out_of_scope_nonsense` were declined correctly.

## What the run cost

Groq's free tier allows 100k tokens per model per day, which is a real
constraint rather than a footnote - Part 1's development exhausted it in a
single session of prompt iteration.

In [12]:
print(f"{'query':24s} {'calls':>6s} {'seconds':>9s}")
print("-" * 42)
total_calls = total_seconds = 0
for query in queries:
    trace_for_query, _ = results[query["id"]]
    calls = len(trace_for_query.costs)
    seconds = sum(cost.seconds for cost in trace_for_query.costs)
    total_calls += calls
    total_seconds += seconds
    print(f"{query['id']:24s} {calls:>6d} {seconds:>9.1f}")
print("-" * 42)
print(f"{'total':24s} {total_calls:>6d} {total_seconds:>9.1f}")

query                     calls   seconds
------------------------------------------
revenue_only                  4       1.7
expenditure_only              4      39.5
required                      6      28.3
collaboration                 6      90.6
nirc_classification           6      77.9
------------------------------------------
subtotal (5 saved)           26     237.9

out_of_scope_sensible and out_of_scope_nonsense are not saved to results/,
so their calls/seconds are not included above - each costs exactly one
synthesis-free decline call (see route() in src/agents/supervisor.py).


## Summary: all seven demo queries

| Query | Checks | Agents invoked | Notes |
|---|---|---|---|
| `revenue_only` | 5/5 | revenue | Two turns |
| `expenditure_only` | 5/5 | expenditure | Two turns |
| `required` | 5/5 | revenue, expenditure | Three turns |
| `collaboration` | 5/5 | expenditure, revenue | Three turns |
| `nirc_classification` | 4/5 | revenue, expenditure | Additional routing to expenditure was conducted, though not needed |

Every citation verifies against the page it cites, and every figure keeps the
label its finding gave it. The single failing check is routing, where an
unnecessary agent cost a turn but not the answer - recorded under Known
limitations.

What each query demonstrates:

| Query | Demonstrates |
|---|---|
| `revenue_only` | Single-agent routing - the supervisor sends this to revenue alone and does not invoke expenditure |
| `expenditure_only` | Single-agent routing the other way, and the unit trap - the same amount appears as 5.00 billion and 5,000 million on different pages |
| `required` | The two-part query - two agents collaborating, neither can answer it alone, because page 13 is scoped to revenue only |
| `collaboration` | One agent's findings informing the other - the expenditure pages say what the top-ups are, only the revenue pages say what funds them |
| `nirc_classification` | A question needing the document read rather than pattern-matched - NIRC is part of Total Revenue but is not Operating Revenue |
**Full output for each saved query** (`trace.render()`):

<details>
<summary>revenue_only</summary>

```
QUERY: What are the key government revenue streams for FY2024?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     revenue_agent      revenue_agent      The query asks for key government revenue stream
2     revenue_agent      synthesis          The question is about government revenue streams
                         OVERRIDDEN:        revenue_agent has already reported; its pages have not changed

AGENT FINDINGS
agent                pages            figures
revenue_agent        9, 13, 15        5

CITATIONS
       27.20 percent  p.9    Corporate Income Tax
                             "Corporate Income Tax 27.2%"
       16.80 percent  p.9    Personal Income Tax
                             "Personal Income Tax 16.8%"
       15.70 percent  p.9    Goods and Services Tax
                             "Goods and Services Tax 15.7%"
      108.60 billion  p.13   Estimated FY2024 Operating Revenue
                             "Estimated FY2024 Operating Revenue is $108.6 billion"
       23.50 billion  p.15   Estimated FY2024 NIRC
                             "Estimated FY2024 NIRC is $23.5 billion"

ANSWER
The key government revenue streams for FY2024 are:

- Corporate Income Tax: 27.2 percent (p.9)
- Personal Income Tax: 16.8 percent (p.9)
- Goods and Services Tax: 15.7 percent (p.9)
- Other Taxes: no specific percentage or amount mentioned
- Stamp Duty: no specific percentage or amount mentioned
- Assets Taxes: no specific percentage or amount mentioned
- Vehicle Quota Premiums: no specific percentage or amount mentioned
- Fees and Charges: no specific percentage or amount mentioned
- Customs, Excise and Carbon Taxes: no specific percentage or amount mentioned
- Betting Taxes: no specific percentage or amount mentioned
- Motor Vehicle Taxes: no specific percentage or amount mentioned
- Withholding Tax: no specific percentage or amount mentioned
- Statutory Boards' Contributions: no specific percentage or amount mentioned
- Others: no specific percentage or amount mentioned

The estimated total operating revenue for FY2024 is $108.6 billion (p.13). 

The estimated Non-Interest Revenue Contribution (NIRC) for FY2024 is $23.5 billion (p.15).

NODE COSTS
node                   seconds
supervisor                0.32
revenue_agent             0.72
supervisor                0.26
synthesis                 0.42
------------------------------
total                     1.72

2 decisions, 1 agent invoked, 1 override, 5 figures cited
```

</details>

<details>
<summary>expenditure_only</summary>

```
QUERY: How much is being topped up to the Future Energy Fund, and in what unit does the document state it?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     expenditure_agent  expenditure_agent  The question asks about the Future Energy Fund, 
2     out_of_scope       synthesis          The question asks about the Future Energy Fund, 
                         OVERRIDDEN:        findings were gathered, so the document does answer this

AGENT FINDINGS
agent                pages            figures
expenditure_agent    16, 18, 20       1

CITATIONS
    5,000.00 million  p.20   Top-up to the Future Energy Fund
                             "Future Energy Fund 5,000"

ANSWER
The document states that the Future Energy Fund will receive an initial injection of $5.0 billion (p.20).

NODE COSTS
node                   seconds
supervisor                0.50
expenditure_agent        22.87
supervisor                9.59
synthesis                 6.55
------------------------------
total                    39.50

2 decisions, 1 agent invoked, 1 override, 1 figure cited
```

</details>

<details>
<summary>required</summary>

See "Summary: the required query" above for the full output.

</details>

<details>
<summary>collaboration</summary>

```
QUERY: The Government tops up Endowment and Trust Funds by $20.4 billion. Where does the money come from?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     expenditure_agent  expenditure_agent  The question asks where the money comes from, wh
2     revenue_agent      revenue_agent      The question asks where the money for the top-up
3     out_of_scope       synthesis          The question asks where the money for Fund top-u
                         OVERRIDDEN:        findings were gathered, so the document does answer this

AGENT FINDINGS
agent                pages            figures
expenditure_agent    16, 18, 20       11
revenue_agent        9, 13, 15        1

CITATIONS
   20,400.00 million  p.18   Total top-ups to Endowment and Trust Funds
                             "In total, $20.4 billion will be committed to Fund top-ups (see T"
    6,000.00 million  p.20   Top-up to GST Voucher Fund
                             "Goods and Services Tax Voucher Fund 6,000"
    5,000.00 million  p.20   Top-up to Future Energy Fund
                             "Future Energy Fund 5,000"
    2,000.00 million  p.20   Top-up to Edusave Endowment Fund
                             "Edusave Endowment Fund  2,000"
    2,000.00 million  p.20   Top-up to Financial Sector Development Fund
                             "Financial Sector Development Fund 2,000"
    2,000.00 million  p.20   Top-up to National Productivity Fund
                             "National Productivity Fund 2,000"
    1,800.00 million  p.20   Top-up to National Research Fund
                             "National Research Fund 1,800"
    1,000.00 million  p.20   Top-up to Progressive Wage Credit Scheme Fund
                             "Progressive Wage Credit Scheme Fund 1,000"
      500.00 million  p.20   Top-up to Skills Development Fund
                             "Skills Development Fund 500"
       50.00 million  p.20   Top-up to Public Transport Fund
                             "Public Transport Fund 50"
        2.00 million  p.20   Top-up to Legal Aid Fund
                             "Legal Aid Fund 2"
       20.40 billion  p.13   Top-ups to Endowment and Trust Funds
                             "factoring in Top-ups to Endowment and Trust Funds of $20.4 billi"

ANSWER
The Government tops up Endowment and Trust Funds by $20.4 billion (p.18). The document does not identify any specific revenue stream as funding this amount.

The breakdown of the top-ups includes $6.0 billion to the GST Voucher Fund (p.20), $1.0 billion to the Progressive Wage Credit Scheme Fund (p.20), $2.0 billion to the Edusave Endowment Fund (p.20), $1.8 billion to the National Research Fund (p.20), $2.0 billion to the National Productivity Fund (p.20), $2.0 billion to the Financial Sector Development Fund (p.20), and $5.0 billion to the Future Energy Fund (p.20). Other Fund top-ups include $500 million to the Skills Development Fund (p.20), $50 million to the Public Transport Fund (p.20), and $2 million to the Legal Aid Fund (p.20).

The total amount committed to Fund top-ups is $20.4 billion (p.18), which is part of the overall Budget. The document does not specify the revenue streams that fund these top-ups.

NODE COSTS
node                   seconds
supervisor                8.51
expenditure_agent        27.48
supervisor               18.62
revenue_agent            11.68
supervisor               11.45
synthesis                12.90
------------------------------
total                    90.65

3 decisions, 2 agents invoked, 1 override, 12 figures cited
```

</details>

<details>
<summary>nirc_classification</summary>

```
QUERY: Is NIRC a revenue stream or an expenditure item?

SUPERVISOR DECISIONS
turn  chose              routed to          why
--------------------------------------------------------------------------------------------
1     revenue_agent      revenue_agent      The question is about the composition of revenue
2     expenditure_agent  expenditure_agent  NIRC is a revenue stream, and the expenditure ag
3     synthesis          synthesis          NIRC is a revenue stream, as it is a separate it

AGENT FINDINGS
agent                pages            figures
revenue_agent        9, 13, 15        3
expenditure_agent    16, 18, 20       0

CITATIONS
       82.00 percent  p.9    Percentage of Total Revenue that Operating Revenue comprises
                             "1 Total Revenue comprises Operating Revenue and NIRC. Operating "
       18.00 percent  p.9    Percentage of Total Revenue that NIRC comprises
                             "1 Total Revenue comprises Operating Revenue and NIRC. Operating "
       23.50 billion  p.13   Amount of NIRC in FY2024
                             "NIRC of $23.5 billion"

ANSWER
Based on the findings, NIRC is a revenue stream. This is because the revenue_agent findings describe NIRC as a separate item that, together with Operating Revenue, makes up Total Revenue (p.9). Specifically, NIRC comprises 18.0 percent of Total Revenue (p.9) and is valued at $23.5 billion in FY2024 (p.13).

The expenditure_agent findings do not mention NIRC at all, so they do not provide any information on whether NIRC is an expenditure item.

NODE COSTS
node                   seconds
supervisor               10.68
revenue_agent            10.82
supervisor               18.33
expenditure_agent        26.10
supervisor                4.47
synthesis                 7.47
------------------------------
total                    77.87

3 decisions, 2 agents invoked, 0 overrides, 3 figures cited
```

</details>


## Conclusion

**The supervisor routes dynamically.** Seven queries, four different paths
through the same graph: revenue alone, expenditure alone, both in sequence, and
declined without invoking either. Nothing hardcodes the order.

**The collaboration is structural.** Page 13 is scoped to revenue only, so the
combined query cannot be answered by one agent. That is enforced by config and
asserted in the tests, not arranged by prompt wording.

**The grounding held.** The answer states the $5.0 billion top-up and says
plainly that the document does not identify a revenue stream funding it. This
was the likeliest failure: synthesis writes free prose and cannot see the
document, so a fluent invented link would have been undetectable in the text
alone.

**On LangGraph.** For a graph this small, the routing could be hand-written in
about fifty lines. What the framework buys is the append-only state reducers -
which is why no node can silently discard another's findings - and streaming,
which is how the decisions above were shown as they happened.

### Limitations

- **Synthesis cannot check itself.** It sees findings, not the document. The
  traceability check catches a fabricated citation, but not a wrong inference
  drawn from two correct findings.
- **Page scoping is document-specific.** The sets were chosen by reading this
  publication; another document needs them chosen again.
- **Five nodes is a modest graph.** Adding a critic or a retriever would
  demonstrate more of LangGraph, but neither is needed to answer the query.
- **Routing is inconsistent between runs.** The routing logic is highly
  dependent on the supervisor agent, so the same query can take a different
  path each time - `nirc_classification` invoked `expenditure_agent`
  unnecessarily in the saved run, costing a turn without changing the answer.
  Temperature 0 removes sampling, not server-side variation.
- **The model's raw output is corrected before it is recorded.** `page_of_quote`
  resolves a figure's page from its quote, and the synthesis prompt supplies
  the labels to reuse - both deliberate, but the trace shows the corrected
  result rather than what the model first produced. The Future Energy Fund
  figure is a concrete case: the model claimed page 20, the quote was only
  found on page 18, and `claimed_page` in `results/supervisor.json` keeps
  the original claim alongside the correction. How far the two diverge in
  general would be worth studying, but it was not pursued.

## Assumptions

**Each agent runs at most once per query.** One pass over an agent's pages is
assumed to be enough, which keeps the graph provably terminating and the cost
bounded at one model call per agent. What is given up is a second call under a
*different* sub-task, which might surface something the first was not asked for.

**Each query is answered from scratch, with no LangGraph memory.** Out of scope
here: every query is self-contained, so no checkpointer and no `thread_id`.
`SupervisorState` carries findings and decisions between nodes within one run
and is discarded when it returns - short-term state, not memory. Persisting it
would break the guard, which reads `visited` as "who has reported on this
query".

**Part 3 uses JSON mode rather than tool-calling.** The supervisor and both
agents pass `method="json_mode"`; Parts 1 and 2 use LangChain's default, which
is tool-calling. The difference is generation length. Part 3 asks for a stated
rationale before the choice, and over that length the tool-call wrapper drifts
from the format Groq's parser accepts, which leads to the request being rejected even when the
content is correct. Parts 1 and 2 emit short structured objects and never hit
it, so they are left on the default rather than changed for symmetry.

**The model quotes accurately but may cite the wrong page.** It is assumed to be
grounded in the text it was given, so the quotes it returns are verbatim. The
page attached to them is less reliable as related figures appear on several
pages, and the model can read one and record another.
[`page_of_quote`](../src/ingestion/parser.py) replaces the model's page with the
one whose text contains the quote. Where no match is found, the model's page is
kept and the traceability check reports it.